# 2_预训练模型_本地模型版_分步运行

来源：https://www.runoob.com/nlp/pre-trained-models.html

页面代码包括 Word2Vec、Transformers 情感分析 Pipeline 和 BERT 分类微调。本 Notebook 保留这些内容，并将在线模型名称改为本地目录加载。

说明：代码已整理为适合 Jupyter Notebook 分段执行的形式；能写在一行的简单语句尽量写在一行。

## 运行环境

```powershell
python -m pip install gensim torch transformers
```

Word2Vec 示例从零训练，不需要下载模型。情感分类和 BERT 微调部分需要本地模型。

In [2]:
import os
import numpy as np
import torch

np.random.seed(42); torch.manual_seed(42)
print("PyTorch版本：",torch.__version__)

PyTorch版本： 2.8.0+cpu


## 1. 页面 Word2Vec 示例：本地训练

In [3]:
from gensim.models import Word2Vec

sentences=[["自然","语言","处理"],["预训练","模型","很强大"],["自然","语言","模型"],["深度","学习","模型"],["我","喜欢","自然","语言","处理"]]
word2vec_model=Word2Vec(sentences,vector_size=100,window=5,min_count=1,workers=1,seed=42,epochs=100)
print("词表：",word2vec_model.wv.index_to_key)
print("“自然”的词向量前10维：",word2vec_model.wv["自然"][:10])

词表： ['模型', '语言', '自然', '处理', '喜欢', '我', '学习', '深度', '很强大', '预训练']
“自然”的词向量前10维： [-2.6855594e-03  8.1963073e-03 -8.5768821e-05  3.9992169e-03
 -8.5277355e-04 -4.6838527e-03  5.2850000e-03  9.3983430e-03
 -4.7486494e-03  5.5697025e-03]


In [4]:
for word,score in word2vec_model.wv.most_similar("自然",topn=5): print(word,round(score,4))

很强大 0.1668
处理 0.1077
深度 0.0647
模型 0.0293
语言 0.0016


### 保存并从本地重新加载 Word2Vec

In [5]:
word2vec_path=r"D:\11\NLP\data\word2vec_local.model"
os.makedirs(os.path.dirname(word2vec_path),exist_ok=True); word2vec_model.save(word2vec_path)
loaded_word2vec=Word2Vec.load(word2vec_path)
print("保存路径：",word2vec_path); print("加载后向量是否一致：",np.allclose(word2vec_model.wv["自然"],loaded_word2vec.wv["自然"]))

保存路径： D:\11\NLP\data\word2vec_local.model
加载后向量是否一致： True


## 2. 本地情感分析 Pipeline

页面原代码：

```python
classifier = pipeline("sentiment-analysis")
```

这会自动联网寻找默认模型。这里改为本地模型目录。

推荐中文二分类模型仓库：`uer/roberta-base-finetuned-jd-binary-chinese`

手动下载方法：

1. 打开 Hugging Face 模型仓库的 **Files and versions**。
2. 下载 `config.json`、分词器文件和一份权重文件。
3. 权重文件选择 `model.safetensors` 或 `pytorch_model.bin`，不要重复下载两份。
4. 把文件直接放入：

```text
D:\11\NLP\data\jd-sentiment-local
```

代码使用 `local_files_only=True`，运行时不会联网。

In [6]:
from transformers import AutoTokenizer,AutoModelForSequenceClassification,pipeline

sentiment_path=r"D:\11\NLP\data\jd-sentiment-local"
required_base=["config.json"]; weight_candidates=["model.safetensors","pytorch_model.bin"]
sentiment_ready=os.path.isdir(sentiment_path) and all(os.path.isfile(os.path.join(sentiment_path,name)) for name in required_base) and any(os.path.isfile(os.path.join(sentiment_path,name)) for name in weight_candidates)
print("情感模型目录：",sentiment_path); print("模型是否准备完成：",sentiment_ready)
if os.path.isdir(sentiment_path): print("目录文件：",os.listdir(sentiment_path))

C:\Users\Administrator\.conda\envs\rl\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


情感模型目录： D:\11\NLP\data\jd-sentiment-local
模型是否准备完成： True
目录文件： ['config.json', 'pytorch_model.bin', 'special_tokens_map.json', 'tokenizer_config.json', 'vocab.txt']


In [7]:
if sentiment_ready:
    sentiment_tokenizer=AutoTokenizer.from_pretrained(sentiment_path,local_files_only=True)
    sentiment_model=AutoModelForSequenceClassification.from_pretrained(sentiment_path,local_files_only=True)
    classifier=pipeline("text-classification",model=sentiment_model,tokenizer=sentiment_tokenizer,device=-1)
    print("模型标签：",sentiment_model.config.id2label)
else: print("请先按照上面的说明，把中文情感分类模型下载到本地目录。")

Device set to use cpu


模型标签： {0: 'negative (stars 1, 2 and 3)', 1: 'positive (stars 4 and 5)'}


In [8]:
texts=["这个产品非常好用，我很满意。","质量很差，刚买回来就坏了。","马云是阿里巴巴的创始人。"]
if sentiment_ready:
    results=classifier(texts,truncation=True)
    for text,result in zip(texts,results): print(text,"->",result)
else: print("模型未准备，跳过情感分类。")

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


这个产品非常好用，我很满意。 -> {'label': 'positive (stars 4 and 5)', 'score': 0.9719884395599365}
质量很差，刚买回来就坏了。 -> {'label': 'negative (stars 1, 2 and 3)', 'score': 0.9838373064994812}
马云是阿里巴巴的创始人。 -> {'label': 'positive (stars 4 and 5)', 'score': 0.9867156147956848}


## 3. 本地 BERT 分类微调

页面原代码使用：

```python
BertForSequenceClassification.from_pretrained("bert-base-chinese")
```

这里改为本地基础 BERT：

```text
D:\11\NLP\data\bert-base-chinese
```

推荐模型仓库：`google-bert/bert-base-chinese`

需要保留：

- `config.json`
- `vocab.txt`
- `tokenizer_config.json`（建议）
- `tokenizer.json`（可选）
- `model.safetensors` 或 `pytorch_model.bin`

In [14]:
from transformers import AutoTokenizer,AutoModelForSequenceClassification

bert_path=r"D:\11\NLP\data\bertbert-base-chinese"
bert_ready=os.path.isdir(bert_path) and os.path.isfile(os.path.join(bert_path,"config.json")) and any(os.path.isfile(os.path.join(bert_path,name)) for name in ["model.safetensors","pytorch_model.bin"])
print("BERT目录：",bert_path); print("BERT是否准备完成：",bert_ready)

BERT目录： D:\11\NLP\data\bertbert-base-chinese
BERT是否准备完成： True


In [15]:
if bert_ready:
    bert_tokenizer=AutoTokenizer.from_pretrained(bert_path,local_files_only=True)
    bert_classifier=AutoModelForSequenceClassification.from_pretrained(bert_path,num_labels=2,local_files_only=True)
    print("分类模型输出类别数：",bert_classifier.config.num_labels)
else: print("请先下载并放置本地中文BERT。")

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at D:\11\NLP\data\bertbert-base-chinese and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


分类模型输出类别数： 2


### 构造一个极小训练数据集，观察输入编码

In [16]:
train_texts=["这个电影很好看","服务非常满意","产品质量太差","体验很糟糕"]
train_labels=torch.tensor([1,1,0,0])
if bert_ready:
    encoded=bert_tokenizer(train_texts,padding=True,truncation=True,max_length=32,return_tensors="pt")
    print("input_ids形状：",encoded["input_ids"].shape); print("第一个样本Token：",bert_tokenizer.convert_ids_to_tokens(encoded["input_ids"][0]))
else: print("模型未准备，跳过编码。")

input_ids形状： torch.Size([4, 9])
第一个样本Token： ['[CLS]', '这', '个', '电', '影', '很', '好', '看', '[SEP]']


### 手动完成一次本地微调，不依赖在线数据集

In [17]:
if bert_ready:
    bert_classifier.train(); optimizer=torch.optim.AdamW(bert_classifier.parameters(),lr=2e-5)
    output=bert_classifier(**encoded,labels=train_labels); print("训练前损失：",output.loss.item())
    optimizer.zero_grad(); output.loss.backward(); optimizer.step()
    output_after=bert_classifier(**encoded,labels=train_labels); print("一次更新后的损失：",output_after.loss.item())
else: print("模型未准备，跳过微调。")

训练前损失： 0.6007510423660278
一次更新后的损失： 0.5605919361114502


### 将微调结果保存到新的本地文件夹

In [18]:
fine_tuned_path=r"D:\11\NLP\data\bert-toy-classifier-local"
if bert_ready:
    bert_classifier.save_pretrained(fine_tuned_path,safe_serialization=True); bert_tokenizer.save_pretrained(fine_tuned_path); print("已保存：",fine_tuned_path)
else: print("模型未准备，没有保存。")

已保存： D:\11\NLP\data\bert-toy-classifier-local
